<a href="https://colab.research.google.com/github/takuya0724/sticker-pro/blob/main/%E3%82%B9%E3%82%BF%E3%83%B3%E3%83%97%E3%83%97%E3%83%ADVersion1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# StickerPro Ultimate v7.1
# Cell① 初期設定・画像読込
# ============================================================

from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import os
import shutil

# ============================================================
# フォルダ初期化
# ============================================================

WORK_FOLDER = "StickerPro"

if os.path.exists(WORK_FOLDER):
    shutil.rmtree(WORK_FOLDER)

os.makedirs(WORK_FOLDER)

TEMP_FOLDER = os.path.join(WORK_FOLDER, "temp")
LINE_FOLDER = os.path.join(WORK_FOLDER, "line")
EXPORT_FOLDER = os.path.join(WORK_FOLDER, "export")

os.makedirs(TEMP_FOLDER)
os.makedirs(LINE_FOLDER)
os.makedirs(EXPORT_FOLDER)

# ============================================================
# 設定
# ============================================================

CONFIG = {
    "grid": 4,
    "transparency": "normal",
    "position": "center",
    "stamp_size": (370, 320),
    "main_size": (240, 240),
    "tab_size": (96, 74),
}

print("=" * 60)
print("StickerPro Ultimate v7.1")
print("=" * 60)

# ============================================================
# 分割数選択
# ============================================================

grid_widget = widgets.RadioButtons(
    options=[
        ("4 × 4（16枚）", 4),
        ("5 × 5（25枚）", 5),
    ],
    value=4,
    description="分割",
)

display(grid_widget)

# ============================================================
# 元画像
# ============================================================

print("\n① 元画像（4×4 または 5×5）を選択してください")

uploaded = files.upload()

if len(uploaded) != 1:
    raise Exception("元画像は1枚だけ選択してください。")

SOURCE_FILENAME = list(uploaded.keys())[0]
SOURCE_IMAGE = Image.open(SOURCE_FILENAME).convert("RGBA")

# ============================================================
# Main画像
# ============================================================

print("\n② Main画像を選択してください")

uploaded = files.upload()

if len(uploaded) != 1:
    raise Exception("Main画像は1枚だけ選択してください。")

MAIN_FILENAME = list(uploaded.keys())[0]
MAIN_IMAGE = Image.open(MAIN_FILENAME).convert("RGBA")

# ============================================================
# TAB画像
# ============================================================

print("\n③ TAB画像を選択してください")

uploaded = files.upload()

if len(uploaded) != 1:
    raise Exception("TAB画像は1枚だけ選択してください。")

TAB_FILENAME = list(uploaded.keys())[0]
TAB_IMAGE = Image.open(TAB_FILENAME).convert("RGBA")

# ============================================================
# 設定保存
# ============================================================

CONFIG["grid"] = grid_widget.value
CONFIG["stamp_count"] = CONFIG["grid"] ** 2

# ============================================================
# プレビュー
# ============================================================

print("\n")
print("=" * 60)
print("設定内容")
print("=" * 60)

print(f"分割　　　：{CONFIG['grid']} × {CONFIG['grid']}")
print(f"スタンプ数：{CONFIG['stamp_count']}枚")
print(f"元画像　　：{SOURCE_IMAGE.width} × {SOURCE_IMAGE.height}")
print(f"Main画像　：{MAIN_IMAGE.width} × {MAIN_IMAGE.height}")
print(f"TAB画像　 ：{TAB_IMAGE.width} × {TAB_IMAGE.height}")

plt.figure(figsize=(6,6))
plt.imshow(SOURCE_IMAGE)
plt.title("元画像")
plt.axis("off")
plt.show()

plt.figure(figsize=(3,3))
plt.imshow(MAIN_IMAGE)
plt.title("Main画像")
plt.axis("off")
plt.show()

plt.figure(figsize=(3,2))
plt.imshow(TAB_IMAGE)
plt.title("TAB画像")
plt.axis("off")
plt.show()

print("✅ Cell① 完了")
print("次は Cell② を実行してください。")

In [ ]:
# ============================================================
# StickerPro Ultimate v7.1
# Cell② 前半
# 分割・背景透過・トリミング準備
# ============================================================

from PIL import Image
import numpy as np
import os

print("=" * 60)
print("スタンプ画像を切り出しています...")
print("=" * 60)

# ============================================================
# tempフォルダ初期化
# ============================================================

if os.path.exists(TEMP_FOLDER):

    import shutil
    shutil.rmtree(TEMP_FOLDER)

os.makedirs(TEMP_FOLDER)

# ============================================================
# 分割サイズ
# ============================================================

GRID = CONFIG["grid"]

piece_width = SOURCE_IMAGE.width // GRID
piece_height = SOURCE_IMAGE.height // GRID

STAMP_PATHS = []
STAMP_IMAGES = []

stamp_no = 1

# ============================================================
# 分割開始
# ============================================================

for row in range(GRID):

    for col in range(GRID):

        left = col * piece_width
        top = row * piece_height
        right = left + piece_width
        bottom = top + piece_height

        img = SOURCE_IMAGE.crop(
            (
                left,
                top,
                right,
                bottom
            )
        ).convert("RGBA")

        # ==========================================
        # numpy変換
        # ==========================================

        data = np.array(img)

        r = data[:, :, 0]
        g = data[:, :, 1]
        b = data[:, :, 2]
        a = data[:, :, 3]

        # ==========================================
        # 白〜薄いグレー背景を透過
        # ==========================================

        mask = (
            (r >= 235)
            &
            (g >= 235)
            &
            (b >= 235)
        )

        a[mask] = 0

        data[:, :, 3] = a

        img = Image.fromarray(data)

        # ==========================================
        # 透明部分を除いてトリミング
        # ==========================================

        bbox = img.getbbox()

        if bbox is not None:

            img = img.crop(bbox)

        # ==========================================
        # 保存ファイル名
        # ==========================================

        filename = f"stamp_{stamp_no:02}.png"

        save_path = os.path.join(
            TEMP_FOLDER,
            filename
        )

        img.save(save_path)

        STAMP_PATHS.append(save_path)
        STAMP_IMAGES.append(img)

        print(f"✓ {filename}")

        stamp_no += 1
# ============================================================
# Cell② 後半
# 処理確認・Cell③へ受け渡し
# ============================================================

# ============================================================
# 作成枚数チェック
# ============================================================

EXPECTED_COUNT = CONFIG["stamp_count"]

if len(STAMP_PATHS) != EXPECTED_COUNT:

    raise Exception(
        f"スタンプ枚数が一致しません "
        f"({len(STAMP_PATHS)} / {EXPECTED_COUNT})"
    )

# ============================================================
# Cell③用
# ============================================================

SELECTED_STAMPS = STAMP_PATHS.copy()

# ============================================================
# 情報表示
# ============================================================

print()
print("=" * 60)
print("切り出し完了")
print("=" * 60)

print(f"分割　　　：{GRID} × {GRID}")
print(f"スタンプ数：{len(STAMP_PATHS)}枚")
print(f"保存先　　：{TEMP_FOLDER}")

print()

for i, path in enumerate(STAMP_PATHS, start=1):

    img = Image.open(path)

    print(
        f"{i:02d} : "
        f"{img.width} × {img.height}"
    )

print()
print("=" * 60)
print("Cell② 完了")
print("次は Cell③ を実行してください。")
print("=" * 60)

In [ ]:
# ============================================================
# StickerPro Ultimate v7.1
# Cell③ スタンプ選択
# ============================================================

import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

print("=" * 60)
print("スタンプ選択")
print("=" * 60)

checkboxes = []
rows = []

COLUMN_COUNT = 4

# ============================================================
# サムネイル作成
# ============================================================

for i, path in enumerate(STAMP_PATHS):

    img = Image.open(path).copy()

    img.thumbnail((120,120))

    buffer = io.BytesIO()
    img.save(buffer, format="PNG")

    image_widget = widgets.Image(
        value=buffer.getvalue(),
        format="png",
        width=120,
        height=120
    )

    checkbox = widgets.Checkbox(
        value=True,
        description=f"{i+1}",
        indent=False
    )

    checkboxes.append(checkbox)

    card = widgets.VBox(
        [
            image_widget,
            checkbox
        ],
        layout=widgets.Layout(
            align_items="center",
            width="130px"
        )
    )

    rows.append(card)

# ============================================================
# 4列表示
# ============================================================

grid = []

for i in range(0, len(rows), COLUMN_COUNT):

    grid.append(

        widgets.HBox(
            rows[i:i+COLUMN_COUNT]
        )

    )

display(widgets.VBox(grid))
# ============================================================
# Cell③ 後半
# ============================================================

output = widgets.Output()

# --------------------------------------------
# 全選択
# --------------------------------------------

btn_all = widgets.Button(
    description="✅ 全て選択",
    button_style="success"
)

# --------------------------------------------
# 全解除
# --------------------------------------------

btn_none = widgets.Button(
    description="❌ 全て解除",
    button_style="warning"
)

# --------------------------------------------
# 決定
# --------------------------------------------

btn_ok = widgets.Button(
    description="▶ 決定",
    button_style="primary"
)

# --------------------------------------------
# 全選択
# --------------------------------------------

def select_all(b):

    for cb in checkboxes:
        cb.value = True

# --------------------------------------------
# 全解除
# --------------------------------------------

def select_none(b):

    for cb in checkboxes:
        cb.value = False

# --------------------------------------------
# 決定
# --------------------------------------------

def finish(b):

    global SELECTED_STAMPS

    SELECTED_STAMPS = []

    for cb, path in zip(checkboxes, STAMP_PATHS):

        if cb.value:
            SELECTED_STAMPS.append(path)

    with output:

        output.clear_output()

        print("=" * 60)
        print("選択完了")
        print("=" * 60)

        print(f"選択枚数：{len(SELECTED_STAMPS)}")

        if len(SELECTED_STAMPS) == 0:

            print("⚠ 1枚以上選択してください。")

        else:

            print("Cell③ 完了")
            print("次は Cell④ を実行してください。")

btn_all.on_click(select_all)
btn_none.on_click(select_none)
btn_ok.on_click(finish)

display(

    widgets.HBox([

        btn_all,
        btn_none,
        btn_ok

    ])

)

display(output)

In [ ]:
# ============================================================
# StickerPro Ultimate v7.1
# Cell④ LINE画像作成
# ============================================================

from PIL import Image
import os

print("=" * 60)
print("LINEスタンプ画像を作成しています...")
print("=" * 60)

# ============================================================
# 出力フォルダ初期化
# ============================================================

if os.path.exists(LINE_FOLDER):

    import shutil
    shutil.rmtree(LINE_FOLDER)

os.makedirs(LINE_FOLDER)

STAMP_SIZE = CONFIG["stamp_size"]
MAIN_SIZE = CONFIG["main_size"]
TAB_SIZE = CONFIG["tab_size"]

# ============================================================
# スタンプ作成
# ============================================================

for i, path in enumerate(SELECTED_STAMPS):

    img = Image.open(path).convert("RGBA")

    canvas = Image.new(
        "RGBA",
        STAMP_SIZE,
        (255,255,255,0)
    )

    ratio = min(

        STAMP_SIZE[0] / img.width,

        STAMP_SIZE[1] / img.height

    ) * 0.90

    new_size = (

        int(img.width * ratio),

        int(img.height * ratio)

    )

    img = img.resize(
        new_size,
        Image.Resampling.LANCZOS
    )

    x = (STAMP_SIZE[0]-img.width)//2

    # ==========================
    # 配置
    # ==========================

    if CONFIG["position"] == "top":

        y = 0

    elif CONFIG["position"] == "bottom":

        y = STAMP_SIZE[1]-img.height

    else:

        y = (STAMP_SIZE[1]-img.height)//2

    canvas.paste(img,(x,y),img)

    save_path = os.path.join(

        LINE_FOLDER,

        f"stamp_{i+1:02}.png"

    )

    canvas.save(save_path)

# ============================================================
# Main画像
# ============================================================

main = MAIN_IMAGE.resize(

    MAIN_SIZE,

    Image.Resampling.LANCZOS

)

main.save(

    os.path.join(

        LINE_FOLDER,

        "main.png"

    )

)

# ============================================================
# TAB画像
# ============================================================

tab = TAB_IMAGE.resize(

    TAB_SIZE,

    Image.Resampling.LANCZOS

)

tab.save(

    os.path.join(

        LINE_FOLDER,

        "tab.png"

    )

)

print()
print("="*60)
print(f"スタンプ : {len(SELECTED_STAMPS)}枚")
print("Main画像 : OK")
print("TAB画像  : OK")
print("="*60)
print("Cell④ 完了")
print("次は Cell⑤ を実行してください。")

In [ ]:
# ============================================================
# StickerPro Ultimate v7.1
# Cell⑤ ZIP作成・ダウンロード
# ============================================================

from google.colab import files
import os
import zipfile

print("=" * 60)
print("ZIPファイルを作成しています...")
print("=" * 60)

# ============================================================
# ZIP保存先
# ============================================================

ZIP_NAME = "StickerPro_LINE_Sticker.zip"
ZIP_PATH = os.path.join(EXPORT_FOLDER, ZIP_NAME)

# 古いZIP削除
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# ============================================================
# ZIP作成
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file in sorted(os.listdir(LINE_FOLDER)):

        path = os.path.join(
            LINE_FOLDER,
            file
        )

        zipf.write(
            path,
            arcname=file
        )

# ============================================================
# 内容表示
# ============================================================

print()
print("=" * 60)
print("ZIP内容")
print("=" * 60)

stamp_count = 0

for file in sorted(os.listdir(LINE_FOLDER)):

    print("✓", file)

    if file.startswith("stamp_"):
        stamp_count += 1

print()

print(f"スタンプ枚数 : {stamp_count}")
print("Main画像     : main.png")
print("TAB画像      : tab.png")

print()
print("=" * 60)
print("ZIP作成完了")
print("=" * 60)

# ============================================================
# ダウンロード
# ============================================================

files.download(ZIP_PATH)

print()
print("🎉 StickerPro Ultimate v7.1 完成！")